In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1mrDvqg_IrU8cDMqiTEUdtWuCsxG9nXVQ", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))

In [ ]:
#@title 🎧 Listen: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# 🔪 Chunking Strategies for RAG: Slicing Documents for Maximum Retrieval Quality

*Part 2 of 5 in the Vizuara series on RAG Systems (Context Engineering Course)*

**Estimated time: 45 minutes**

In the previous notebook, we built a basic RAG pipeline and saw how retrieval-augmented generation works end to end. But here is a dirty secret of production RAG systems: **the single biggest lever you have for improving retrieval quality is not the embedding model, not the vector database, and not the reranker. It is how you chunk your documents.**

Imagine trying to find a specific recipe in a cookbook where someone photocopied random pages and shuffled them — that is what bad chunking does to your retrieval. A perfectly good embedding model and a lightning-fast vector database will still return garbage if the chunks themselves do not make sense.

By the end of this notebook, you will have implemented four chunking strategies from scratch, understood the mathematics of why chunking matters, and built an interactive tool to compare them side by side on real text.

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/rag-systems/practice/2/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Installations
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_installations.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 0. 🛠️ Setup and Installations

Let us install the libraries we need. We use `sentence-transformers` for embedding text (this is a **tool**, not the concept we are learning), `langchain-text-splitters` for the recursive chunking baseline, and `tiktoken` for accurate token counting.

In [ ]:
!pip install -q sentence-transformers numpy matplotlib langchain-text-splitters tiktoken requests

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import textwrap
import re
import time
import requests
from typing import List, Tuple, Dict, Optional

# Reproducibility
np.random.seed(42)

# Plotting defaults
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.facecolor'] = '#fafafa'

print("Setup complete!")

In [ ]:
#@title 🎧 Listen: Why It Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_why_it_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. 🤔 Why Does This Matter?

Chunking is the most underrated step in the entire RAG pipeline. Engineers spend weeks tuning embedding models and rerankers, but give their chunking strategy about five minutes of thought. This is backwards.

Here is why chunking matters so much:

| What Goes Wrong | Impact on Retrieval | Impact on Generation |
|---|---|---|
| Chunks too large (2000+ tokens) | Embedding averages over too many topics; similarity becomes imprecise | LLM gets irrelevant context mixed with relevant context |
| Chunks too small (50 tokens) | Each chunk lacks enough context to be meaningful | LLM cannot understand the chunk without surrounding information |
| Splits at bad boundaries | Key facts get torn in half across two chunks | Neither chunk alone contains the complete answer |
| No overlap between chunks | Information at boundaries is lost forever | The exact sentence the user needs lives in a gap |

The stakes are high. A study by Pinecone (2023) found that changing chunk size alone could swing retrieval Recall@5 by **up to 20 percentage points** — a bigger effect than switching embedding models.

Let us make this concrete. Here is a passage that gets destroyed by bad chunking:

In [ ]:
#@title 🎧 Listen: Chunking Failure Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_chunking_failure_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# The full passage
full_passage = (
    "The transformer architecture processes sequences using self-attention, "
    "which computes attention weights between every pair of tokens. "
    "This mechanism was introduced in the 2017 paper 'Attention Is All You Need' "
    "by Vaswani et al."
)

# A naive split right in the middle
midpoint = full_passage.index("This mechanism")
chunk_a = full_passage[:midpoint].strip()
chunk_b = full_passage[midpoint:].strip()

query = "Who introduced the self-attention mechanism?"

print("=" * 70)
print("THE CHUNKING FAILURE")
print("=" * 70)
print(f"\nQuery: \"{query}\"\n")
print(f"Chunk A: \"{chunk_a}\"")
print(f"  -> Contains 'self-attention'? {'self-attention' in chunk_a.lower()}")
print(f"  -> Contains 'Vaswani'?       {'vaswani' in chunk_a.lower()}")
print(f"\nChunk B: \"{chunk_b}\"")
print(f"  -> Contains 'self-attention'? {'self-attention' in chunk_b.lower()}")
print(f"  -> Contains 'Vaswani'?       {'vaswani' in chunk_b.lower()}")
print(f"\nNeither chunk alone can answer the question!")
print(f"The fact is split across the boundary.")

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. 💡 Building Intuition

### The Goldilocks Problem

Think of chunking as Goldilocks trying to find the right bowl of porridge:

**Too large (Papa Bear's bowl):** A 2000-token chunk about "the history of neural networks" covers attention mechanisms, backpropagation, convolutional networks, and recurrent architectures all in one chunk. When the user asks "How does self-attention work?", this chunk matches — but 80% of its content is irrelevant. The embedding is a diluted average of too many topics. Worse, the LLM receives a wall of text where the relevant sentences are buried.

**Too small (Baby Bear's bowl):** A 30-token chunk says: "The attention weights are computed using the softmax function." That is technically correct, but it lacks the context to explain *what* is being attended to, *why* softmax is used, or *how* this fits into the transformer architecture. The embedding captures a narrow meaning, and the LLM cannot construct a useful answer from this fragment alone.

**Just right (Mama Bear's bowl):** A 200-500 token chunk covers one coherent idea: "Self-attention in transformers works by computing query, key, and value vectors for each token. The attention weights are the softmax of the dot product of queries and keys, scaled by the square root of the dimension. These weights determine how much each token attends to every other token in the sequence." This is self-contained, topically focused, and gives the LLM enough context to generate a good answer.

### Overlap: Insurance Against Information Splitting

Even with perfect chunk sizes, you face a fundamental problem: **information at boundaries gets split.** If a key fact starts at position 480 and ends at position 520, and your chunk boundary is at position 500, that fact is torn in half. Neither chunk alone contains it.

Overlap is your insurance policy. By repeating the last $m$ tokens of chunk $i$ at the beginning of chunk $i+1$, you ensure that any fact of length $\leq m$ that spans a boundary appears in full in at least one chunk.

But overlap comes at a cost: more chunks means more embeddings to compute, more storage, and more candidates to search through. And heavily overlapping chunks can dilute retrieval by creating near-duplicate entries. The sweet spot is typically **10-20% overlap** — enough to catch boundary-spanning facts without creating excessive redundancy.

### The Four Strategies We Will Build

1. **Fixed-size chunking** — Split every $n$ tokens, simple but blind to structure
2. **Sentence-level chunking** — Split at sentence boundaries, respects linguistic units
3. **Recursive chunking** — Hierarchical: try paragraphs first, then sentences, then fixed-size
4. **Semantic chunking** — Use embeddings to detect topic shifts, the most intelligent approach

Each has different trade-offs. Let us formalize them mathematically, then build them all.

In [ ]:
#@title 🎧 Listen: The Mathematics Splitting
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_the_mathematics_splitting.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. 📐 The Mathematics

### Information Loss at Chunk Boundaries

Let us formalize the boundary-splitting problem. Suppose we have a document of length $L$ tokens, and we split it into chunks of size $C$ tokens with no overlap. The number of chunk boundaries is:

$$B = \left\lfloor \frac{L}{C} \right\rfloor - 1$$

**What does this mean computationally?** For a 10,000-token document with 500-token chunks, we have $B = 10000/500 - 1 = 19$ boundaries. Each boundary is a potential site where information gets split.

Now suppose a "key fact" in the document has average length $f$ tokens (a typical fact or statement is 20-80 tokens). The probability that a randomly positioned fact of length $f$ spans at least one boundary is:

$$P(\text{split}) = \frac{f}{C}$$

**The computational interpretation:** If facts average $f = 40$ tokens and chunks are $C = 500$ tokens, then $P(\text{split}) = 40/500 = 0.08$, meaning 8% of facts get split. With $C = 100$ token chunks, $P(\text{split}) = 40/100 = 0.40$ — a staggering 40% of facts are torn apart. This is why very small chunks are dangerous even though they seem more precise.

In [ ]:
#@title 🎧 Listen: Viz Splitting Prob
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_viz_splitting_prob.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualize: Probability of fact splitting vs chunk size
fact_lengths = [20, 40, 60, 80]  # typical fact lengths in tokens
chunk_sizes = np.arange(50, 1001, 10)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

for f, color in zip(fact_lengths, colors):
    p_split = f / chunk_sizes
    ax.plot(chunk_sizes, p_split * 100, label=f'Fact length = {f} tokens',
            color=color, linewidth=2.5)

ax.axhline(y=10, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.text(850, 11, '10% threshold', fontsize=10, color='gray')

ax.set_xlabel('Chunk Size (tokens)', fontsize=13)
ax.set_ylabel('Probability of Fact Being Split (%)', fontsize=13)
ax.set_title('The Boundary Splitting Problem:\nSmaller Chunks = More Split Facts', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.set_xlim(50, 1000)
ax.set_ylim(0, 85)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print concrete numbers
print("Concrete example: Document with 10,000 tokens")
print("-" * 55)
for C in [100, 200, 500, 1000]:
    B = (10000 // C) - 1
    p_split_40 = 40 / C
    expected_splits = p_split_40 * B
    print(f"  Chunk size {C:>4}: {B:>3} boundaries, "
          f"P(split) = {p_split_40:.1%}, "
          f"~{expected_splits:.1f} facts split per doc")

In [ ]:
#@title 🎧 Listen: The Mathematics Overlap Shannon
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_the_mathematics_overlap_shannon.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Overlap Recovery Formula

Overlap rescues split facts. With overlap $O$ tokens, a fact of length $f$ that spans a boundary is fully recovered in one of the overlapping chunks if $f \leq O + 1$. The probability of a fact being **irrecoverably** split becomes:

$$P(\text{irrecoverable split}) = \max\left(0, \frac{f - O}{C}\right)$$

**Computational interpretation:** With $C = 500$, $O = 50$, and $f = 40$: since $f < O$, no facts of this length are ever lost — the overlap is sufficient insurance. But for $f = 80$: $P = (80 - 50)/500 = 0.06$, meaning 6% of longer facts still get split. This tells us exactly how much overlap we need for our expected fact length.

The **boundary preservation ratio** is:

$$\text{BPR}(f, O) = \min\left(1, \; \frac{O}{f}\right)$$

**Translation:** For our example with $O = 50$ tokens, any fact shorter than 50 tokens that falls at a boundary will be fully preserved in at least one chunk. Most sentences are 15-30 tokens, so 10-20% overlap covers most boundary sentences.

### Shannon's Information Theory Connection

Claude Shannon showed that information content depends on structure — the entropy of a sequence depends on the statistical dependencies between symbols. When we chunk text at arbitrary positions (fixed-size chunking), we are treating the text as if it has no structure — we are maximizing the entropy of the chunk boundaries.

Sentence boundaries, paragraph breaks, and topic shifts are low-entropy boundary points — they are where the text's own structure naturally segments. Chunking at these points preserves the mutual information between tokens within a chunk, because tokens that are semantically related stay together.

Formally, for a good chunk $c_i$, we want high **intra-chunk mutual information**:

$$I(\text{tokens within } c_i) \gg I(\text{tokens across } c_i, c_{i+1})$$

**Translation:** The tokens inside a chunk should be highly related to each other (high mutual information), while tokens in adjacent chunks should be relatively independent (low mutual information). Sentence and paragraph boundaries are natural minimizers of cross-boundary mutual information.

This is exactly what semantic chunking tries to achieve algorithmically — find the boundaries where the "information flow" between consecutive sentences drops.

In [ ]:
#@title 🎧 Listen: Viz Overlap Cost Bpr
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_viz_overlap_cost_bpr.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualize: Overlap recovery — BPR vs overlap for different fact lengths
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Number of chunks vs chunk size for different overlaps
doc_length = 5000  # tokens
chunk_sizes_plot = np.arange(100, 1001, 50)
overlap_fractions = [0.0, 0.1, 0.2, 0.3]
colors_ov = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

ax1 = axes[0]
for frac, color in zip(overlap_fractions, colors_ov):
    num_chunks = []
    for s in chunk_sizes_plot:
        o = int(s * frac)
        stride = max(s - o, 1)
        k = int(np.ceil((doc_length - o) / stride))
        num_chunks.append(k)
    ax1.plot(chunk_sizes_plot, num_chunks, 'o-', color=color, linewidth=2,
             markersize=4, label=f'{int(frac*100)}% overlap')

ax1.set_xlabel('Chunk Size (tokens)', fontsize=12)
ax1.set_ylabel('Number of Chunks', fontsize=12)
ax1.set_title('Overlap Cost: More Overlap = More Chunks\n(5000-token document)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Right: BPR vs overlap for different fact lengths
ax2 = axes[1]
overlaps = np.arange(0, 105, 5)
fact_lengths_bpr = [10, 20, 40, 60]
fact_colors = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6']

for f_len, color in zip(fact_lengths_bpr, fact_colors):
    bpr = np.minimum(1.0, overlaps / f_len)
    ax2.plot(overlaps, bpr, '-', color=color, linewidth=2.5,
             label=f'Fact length = {f_len} tokens')

ax2.set_xlabel('Overlap (tokens)', fontsize=12)
ax2.set_ylabel('Boundary Preservation Ratio', fontsize=12)
ax2.set_title('Overlap Preserves Boundary Information', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-0.05, 1.1)
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print("Left: As chunk size grows, fewer chunks are needed. Overlap increases the count slightly.")
print("Right: A 40-token overlap preserves 100% of facts up to 40 tokens long.")
print("       Most sentences are 15-30 tokens, so 10-20% overlap is usually sufficient.")

In [ ]:
#@title 🎧 Listen: Loading Document
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_loading_document.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. 🔧 Let Us Build It — Component by Component

### 4.1 Loading a Real Document

Let us work with a real, substantial document — the Wikipedia article about the Transformer architecture. This gives us enough text to see meaningful differences between chunking strategies.

In [ ]:
# Fetch the Wikipedia article about the Transformer model
url = "https://en.wikipedia.org/w/api.php"
params = {
    "action": "query",
    "titles": "Transformer (deep learning architecture)",
    "prop": "extracts",
    "explaintext": True,
    "format": "json"
}

print("Fetching Wikipedia article on Transformers...")
response = requests.get(url, params=params)
data = response.json()

# Extract the text content
pages = data['query']['pages']
page_id = list(pages.keys())[0]

if page_id == '-1':
    # Fallback: try alternative title
    params["titles"] = "Transformer (machine learning model)"
    response = requests.get(url, params=params)
    data = response.json()
    pages = data['query']['pages']
    page_id = list(pages.keys())[0]

raw_text = pages[page_id].get('extract', '')

# If Wikipedia API fails, use a substantial fallback text
if not raw_text or len(raw_text) < 1000:
    raw_text = """The Transformer is a deep learning architecture based on the multi-head attention mechanism, proposed in the 2017 paper "Attention Is All You Need". Text is converted to numerical representations called tokens, and each token is converted into a vector via looking up from a word embedding table. At each layer, each token is then contextualized within the scope of the context window with other (unmasked) tokens via a parallel multi-head attention mechanism allowing the signal for key tokens to be amplified and less important tokens to be diminished.

Transformers have the advantage of having no recurrent units, and therefore require less training time than earlier recurrent neural architectures such as long short-term memory (LSTM). Later variations have been widely adopted for training large language models (LLM) on large (language) datasets, such as the Wikipedia corpus and Common Crawl.

The transformer architecture consists of an encoder and a decoder, each of which is composed of a series of identical layers. Each layer in the encoder contains two sublayers: a multi-head self-attention mechanism and a position-wise fully connected feed-forward network. Each layer in the decoder contains three sublayers: a masked multi-head self-attention mechanism, a multi-head attention mechanism that attends to the encoder output, and a position-wise fully connected feed-forward network.

The self-attention mechanism is the key innovation. For each input token, three vectors are computed: a query vector, a key vector, and a value vector. The attention weight between two tokens is computed as the dot product of the query vector of one token with the key vector of another, divided by the square root of the dimension of the key vectors. These weights are then normalized using the softmax function and used to compute a weighted sum of the value vectors.

Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. Instead of performing a single attention function, the queries, keys and values are linearly projected h times with different learned projections. The attention function is performed in parallel on each of these projected versions, and the results are concatenated and once again projected.

The position-wise feed-forward network consists of two linear transformations with a ReLU activation in between. While the linear transformations are the same across different positions, they use different parameters from layer to layer. This can also be described as two convolutions with kernel size 1.

Since the model contains no recurrence and no convolution, positional information is injected using positional encodings. These are added to the input embeddings at the bottom of the encoder and decoder stacks. The original transformer used sinusoidal positional encodings, where the position is encoded using sine and cosine functions of different frequencies.

The transformer was originally designed for sequence-to-sequence tasks such as machine translation. The encoder processes the input sequence and produces a set of continuous representations. The decoder then generates the output sequence one token at a time, attending to both the encoder output and the previously generated tokens.

Training is performed using teacher forcing, where the correct output tokens are fed as input to the decoder during training, rather than the decoder's own predictions. The loss function is the standard cross-entropy loss between the predicted and actual output tokens.

The original "Attention Is All You Need" paper demonstrated state-of-the-art results on English-to-German and English-to-French translation tasks. The model achieved a BLEU score of 28.4 on the WMT 2014 English-to-German translation task, improving over the existing best results by more than 2 BLEU.

Since its introduction, the transformer architecture has been adapted for a wide range of tasks beyond machine translation. BERT (Bidirectional Encoder Representations from Transformers) uses only the encoder part and is pre-trained using masked language modeling and next sentence prediction. GPT (Generative Pre-trained Transformer) uses only the decoder part and is pre-trained using autoregressive language modeling. T5 treats every NLP task as a text-to-text problem and uses the full encoder-decoder architecture.

Vision Transformers (ViT) adapt the transformer for image classification by splitting images into patches and treating each patch as a token. This approach has shown that transformers can match or exceed the performance of convolutional neural networks on image tasks when trained on sufficient data.

The scaling properties of transformers have been a major area of research. The scaling laws discovered by Kaplan et al. show that model performance improves predictably with increases in model size, dataset size, and compute budget. This has driven the development of increasingly large models, from BERT's 340 million parameters to GPT-3's 175 billion parameters and beyond.

Efficient transformer variants have been developed to address the quadratic complexity of self-attention with respect to sequence length. Linear attention mechanisms, sparse attention patterns, and hierarchical approaches have been proposed to enable transformers to process longer sequences more efficiently. FlashAttention uses tiling and recomputation to achieve significant speedups on GPU hardware without approximating the attention computation.

The transformer's success has led to its adoption across many domains beyond natural language processing. In computer vision, audio processing, protein structure prediction, weather forecasting, and robotics, transformer-based models have achieved state-of-the-art results. The architecture's flexibility and scalability have made it the dominant paradigm in modern deep learning research."""

print(f"Document loaded: {len(raw_text)} characters")
print(f"Approximate word count: ~{len(raw_text.split())}")
print(f"\nFirst 300 characters:")
print(raw_text[:300])
print("...")

In [ ]:
#@title 🎧 Listen: Fixed Size Chunking
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_fixed_size_chunking.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.2 Fixed-Size Chunking from Scratch

The simplest strategy: split text into chunks of a fixed number of tokens (we use words as a proxy) with configurable overlap. Fast and predictable, but completely blind to sentence and paragraph boundaries.

In [ ]:
import tiktoken

# Load tokenizer for accurate token counting
try:
    tokenizer = tiktoken.get_encoding("cl100k_base")  # GPT-4 tokenizer
    def count_tokens(text: str) -> int:
        return len(tokenizer.encode(text))
    print("Using tiktoken cl100k_base tokenizer for accurate token counts")
except Exception:
    def count_tokens(text: str) -> int:
        return len(text.split())  # Fallback to word count
    print("Using word-count approximation for token counts")


def chunk_fixed_size(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """
    Fixed-size chunking with configurable overlap.

    Args:
        text: The document text to chunk
        chunk_size: Target number of words per chunk
        overlap: Number of words to overlap between consecutive chunks

    Returns:
        List of text chunks
    """
    words = text.split()
    chunks = []
    stride = chunk_size - overlap
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk.strip())
        start += stride

    return chunks


# Apply to our document
fixed_chunks = chunk_fixed_size(raw_text, chunk_size=500, overlap=50)

print(f"Fixed-size chunking (500 words, 50 overlap):")
print(f"  Total chunks: {len(fixed_chunks)}")
print(f"  Chunk sizes (words): {[len(c.split()) for c in fixed_chunks]}")
print(f"\n--- Chunk 1 (first 200 chars) ---")
print(fixed_chunks[0][:200])
if len(fixed_chunks) > 1:
    print(f"\n--- Chunk 2 (first 200 chars) ---")
    print(fixed_chunks[1][:200])

In [ ]:
#@title 🎧 Listen: Viz Fixed Size
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_viz_fixed_size.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualize: Chunk size distribution and split locations for fixed-size chunking
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Bar chart of chunk sizes
chunk_word_counts = [len(c.split()) for c in fixed_chunks]
axes[0].bar(range(len(chunk_word_counts)), chunk_word_counts,
            color='#2196F3', alpha=0.8, edgecolor='white')
axes[0].axhline(y=500, color='red', linestyle='--', alpha=0.6, label='Target size (500)')
axes[0].set_xlabel('Chunk Index', fontsize=12)
axes[0].set_ylabel('Words per Chunk', fontsize=12)
axes[0].set_title('Fixed-Size Chunking: Chunk Sizes', fontsize=14, fontweight='bold')
axes[0].legend()

# Right: Text heatmap showing where splits occur
text_len = len(raw_text)
split_positions = []
pos = 0
for chunk in fixed_chunks[:-1]:
    pos += len(chunk)
    split_positions.append(pos / text_len)

ax2 = axes[1]
n_segments = 100
segment_colors = np.zeros(n_segments)
for sp in split_positions:
    idx = int(sp * n_segments)
    if idx < n_segments:
        segment_colors[idx] = 1

cmap = LinearSegmentedColormap.from_list('split', ['#E3F2FD', '#F44336'])
for i in range(n_segments):
    ax2.barh(0, 1/n_segments, left=i/n_segments, height=0.4,
             color=cmap(segment_colors[i]), edgecolor='none')

ax2.set_xlim(0, 1)
ax2.set_ylim(-0.5, 0.5)
ax2.set_yticks([])
ax2.set_xlabel('Document Position (0 = start, 1 = end)', fontsize=12)
ax2.set_title('Split Locations in Document\n(Red = split boundary)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Notice: Fixed-size splits are perfectly evenly spaced — no awareness of content.")

In [ ]:
#@title 🎧 Listen: Sentence Level Chunking
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_sentence_level_chunking.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.3 Sentence-Level Chunking

Instead of splitting at arbitrary positions, we split at sentence boundaries using regex. This ensures no sentence is ever torn in half, producing more coherent chunks.

In [ ]:
def chunk_by_sentences(text: str, max_chunk_tokens: int = 500) -> List[str]:
    """
    Sentence-level chunking: split at sentence boundaries and group
    sentences until the target chunk size is reached.

    Args:
        text: The document text to chunk
        max_chunk_tokens: Maximum number of words per chunk

    Returns:
        List of text chunks, each ending at a sentence boundary
    """
    # Split on sentence-ending punctuation followed by whitespace
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    current_chunk = []
    current_size = 0

    for sentence in sentences:
        sentence_words = len(sentence.split())

        # If adding this sentence would exceed the limit, finalize current chunk
        if current_size + sentence_words > max_chunk_tokens and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_size = 0

        current_chunk.append(sentence)
        current_size += sentence_words

    # Do not forget the last chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


# Apply to our document
sentence_chunks = chunk_by_sentences(raw_text, max_chunk_tokens=500)

print(f"Sentence-level chunking (max 500 words per chunk):")
print(f"  Total chunks: {len(sentence_chunks)}")
print(f"  Chunk sizes (words): {[len(c.split()) for c in sentence_chunks]}")

# Show a chunk boundary to demonstrate clean splits
if len(sentence_chunks) >= 2:
    print(f"\n--- End of Chunk 1 ---")
    print(f"...{sentence_chunks[0][-150:]}")
    print(f"\n--- Start of Chunk 2 ---")
    print(f"{sentence_chunks[1][:150]}...")
    print(f"\nNotice: clean split at a sentence boundary — no torn sentences!")

In [ ]:
#@title 🎧 Listen: Viz Sentence Level
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_viz_sentence_level.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualize: Compare fixed-size vs sentence-level chunk size distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fixed-size histogram
fixed_sizes = [len(c.split()) for c in fixed_chunks]
axes[0].hist(fixed_sizes, bins=max(5, len(set(fixed_sizes))),
             color='#2196F3', alpha=0.8, edgecolor='white')
axes[0].axvline(x=np.mean(fixed_sizes), color='red', linestyle='--',
                label=f'Mean = {np.mean(fixed_sizes):.0f}')
axes[0].set_xlabel('Words per Chunk', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Fixed-Size Chunks', fontsize=14, fontweight='bold')
axes[0].legend()

# Sentence-level histogram
sent_sizes = [len(c.split()) for c in sentence_chunks]
axes[1].hist(sent_sizes, bins=max(5, len(set(sent_sizes))),
             color='#4CAF50', alpha=0.8, edgecolor='white')
axes[1].axvline(x=np.mean(sent_sizes), color='red', linestyle='--',
                label=f'Mean = {np.mean(sent_sizes):.0f}')
axes[1].set_xlabel('Words per Chunk', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Sentence-Level Chunks', fontsize=14, fontweight='bold')
axes[1].legend()

plt.suptitle('Chunk Size Distributions: Fixed vs Sentence-Level',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Fixed-size:     mean={np.mean(fixed_sizes):.0f}, std={np.std(fixed_sizes):.0f}, "
      f"min={min(fixed_sizes)}, max={max(fixed_sizes)}")
print(f"Sentence-level: mean={np.mean(sent_sizes):.0f}, std={np.std(sent_sizes):.0f}, "
      f"min={min(sent_sizes)}, max={max(sent_sizes)}")

In [ ]:
#@title 🎧 Listen: Recursive Chunking
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_recursive_chunking.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.4 Recursive Chunking

Recursive chunking is the strategy used by LangChain's `RecursiveCharacterTextSplitter`. It tries to split hierarchically: first at paragraph boundaries (`\n\n`), then at newlines (`\n`), then at sentence boundaries (`. `), then at word boundaries (` `), and finally at character level as a last resort. This preserves the most natural structure possible.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the recursive splitter
# We target ~500 tokens per chunk. Since 1 token ~ 4 chars, we use 2000 chars.
# Overlap of 50 tokens ~ 200 chars.
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,        # ~500 tokens worth of characters
    chunk_overlap=200,      # ~50 tokens of overlap
    length_function=len,    # Use character length
    separators=["\n\n", "\n", ". ", " ", ""],  # Try these in order
    is_separator_regex=False,
)

# Apply to our document
recursive_chunks = recursive_splitter.split_text(raw_text)

print(f"Recursive chunking (LangChain RecursiveCharacterTextSplitter):")
print(f"  Total chunks: {len(recursive_chunks)}")
print(f"  Chunk sizes (chars): {[len(c) for c in recursive_chunks]}")
print(f"  Chunk sizes (words): {[len(c.split()) for c in recursive_chunks]}")

# Show a chunk to see the quality of splits
if recursive_chunks:
    print(f"\n--- Chunk 1 (first 300 chars) ---")
    print(recursive_chunks[0][:300])
    if len(recursive_chunks) > 1:
        print(f"\n--- Chunk 2 (first 300 chars) ---")
        print(recursive_chunks[1][:300])

In [ ]:
#@title 🎧 Listen: Semantic Chunking
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_semantic_chunking.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.5 Semantic Chunking

The most sophisticated approach: use sentence embeddings to detect **topic shifts** within the document. When the cosine similarity between consecutive sentences drops below a threshold, we place a chunk boundary there.

This is the only strategy that actually "understands" the content — it groups sentences about the same topic together, regardless of their position or length.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load embedding model
print("Loading embedding model (first run downloads ~90MB)...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded! Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")


def cosine_similarity_pair(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


def chunk_semantic(text: str, model, threshold: float = 0.5,
                   min_chunk_size: int = 2) -> Tuple[List[str], List[float]]:
    """
    Semantic chunking: split where cosine similarity between
    consecutive sentence embeddings drops below a threshold.

    Args:
        text: The document text to chunk
        model: SentenceTransformer model for embeddings
        threshold: Similarity threshold for splitting (lower = fewer, larger chunks)
        min_chunk_size: Minimum number of sentences per chunk

    Returns:
        Tuple of (chunks, similarities) where similarities are the
        cosine similarities between consecutive sentences
    """
    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip() and len(s.strip()) > 10]

    if len(sentences) <= 1:
        return [text], []

    # Embed all sentences
    print(f"  Embedding {len(sentences)} sentences...")
    embeddings = model.encode(sentences, show_progress_bar=False)

    # Compute cosine similarity between consecutive sentences
    similarities = []
    for i in range(len(sentences) - 1):
        sim = cosine_similarity_pair(embeddings[i], embeddings[i + 1])
        similarities.append(sim)

    # Split where similarity drops below threshold
    chunks = []
    current_chunk = [sentences[0]]
    sentence_count = 1

    for i in range(1, len(sentences)):
        sim = similarities[i - 1]

        # Split if similarity is below threshold AND we have enough sentences
        if sim < threshold and sentence_count >= min_chunk_size:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
            sentence_count = 1
        else:
            current_chunk.append(sentences[i])
            sentence_count += 1

    # Add the last chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks, similarities


# Apply to our document
print("Running semantic chunking...")
semantic_chunks, sentence_sims = chunk_semantic(
    raw_text, embed_model, threshold=0.35, min_chunk_size=3
)

print(f"\nSemantic chunking (threshold=0.35):")
print(f"  Total chunks: {len(semantic_chunks)}")
print(f"  Chunk sizes (words): {[len(c.split()) for c in semantic_chunks]}")
print(f"  Avg similarity between consecutive sentences: {np.mean(sentence_sims):.3f}")

In [ ]:
#@title 🎧 Listen: Viz Semantic Sim
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_viz_semantic_sim.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization: Semantic similarity between consecutive sentences
# with split points highlighted

fig, ax = plt.subplots(figsize=(14, 5))

x = range(len(sentence_sims))
colors_bar = ['#F44336' if s < 0.35 else '#4CAF50' for s in sentence_sims]

ax.bar(x, sentence_sims, color=colors_bar, alpha=0.8, width=1.0, edgecolor='none')
ax.axhline(y=0.35, color='red', linestyle='--', linewidth=2,
           label='Split threshold (0.35)')

ax.set_xlabel('Sentence Pair Index (sentence i vs sentence i+1)', fontsize=12)
ax.set_ylabel('Cosine Similarity', fontsize=12)
ax.set_title('Semantic Chunking: Similarity Between Consecutive Sentences\n'
             '(Red bars = topic shift detected, chunk boundary placed here)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print the detected topic shifts
below_threshold = [(i, s) for i, s in enumerate(sentence_sims) if s < 0.35]
print(f"Detected {len(below_threshold)} topic shifts (similarity < 0.35):")
for idx, sim in below_threshold[:8]:
    print(f"  Between sentence {idx} and {idx+1}: similarity = {sim:.3f}")

In [ ]:
#@title 🎧 Listen: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. ✍️ Your Turn (TODO Sections)

### 🔧 TODO 1: Chunk Quality Scorer — Measuring Self-Containedness

A good chunk should be **self-contained** — it should make sense on its own, without needing the surrounding context. We can measure this using **embedding coherence**: how similar is each sentence in the chunk to the chunk's overall meaning?

A high-quality chunk has sentences that all point in the same direction in embedding space. A low-quality chunk is a grab-bag of unrelated sentences that happen to be near each other in the document.

In [ ]:
#@title 🎧 Listen: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def score_chunk_quality(chunk: str, model) -> Dict[str, float]:
    """
    Score the quality of a chunk based on self-containedness.

    A good chunk has high internal coherence — all its sentences
    are about the same topic and make sense together.

    Args:
        chunk: The text chunk to evaluate
        model: SentenceTransformer model for embeddings

    Returns:
        Dictionary with:
        - 'coherence': Average cosine similarity of each sentence
                       to the chunk centroid (0 to 1, higher = better)
        - 'min_similarity': Minimum sentence-to-centroid similarity
                           (identifies outlier sentences)
        - 'self_contained_score': Combined quality score (0 to 1)

    Hints:
        1. Split the chunk into sentences
        2. Embed each sentence AND the full chunk
        3. Compute the centroid (mean) of sentence embeddings
        4. Measure each sentence's cosine similarity to the centroid
        5. The coherence score is the mean of these similarities
        6. The self-contained score combines coherence with a
           penalty for very short chunks (< 3 sentences)
    """
    # ============ TODO ============
    # Step 1: Split chunk into sentences
    sentences = re.split(r'(?<=[.!?])\s+', chunk.strip())
    sentences = [s for s in sentences if len(s.strip()) > 10]

    # Handle edge cases
    if len(sentences) <= 1:
        return {'coherence': 1.0, 'min_similarity': 1.0, 'self_contained_score': 0.5}

    # Step 2: Embed all sentences
    # sentence_embeddings = model.encode(sentences)  # YOUR CODE HERE

    # Step 3: Compute centroid (mean of all sentence embeddings)
    # centroid = np.mean(sentence_embeddings, axis=0)  # YOUR CODE HERE

    # Step 4: Compute cosine similarity of each sentence to centroid
    # similarities = []
    # for emb in sentence_embeddings:
    #     sim = np.dot(emb, centroid) / (np.linalg.norm(emb) * np.linalg.norm(centroid) + 1e-10)
    #     similarities.append(sim)
    # similarities = np.array(similarities)  # YOUR CODE HERE

    # Step 5: Compute coherence (mean similarity) and min similarity
    # coherence = float(np.mean(similarities))  # YOUR CODE HERE
    # min_sim = float(np.min(similarities))     # YOUR CODE HERE

    # Step 6: Compute self-contained score
    # Penalize very short chunks: multiply by min(1, len(sentences) / 3)
    # length_factor = min(1.0, len(sentences) / 3.0)  # YOUR CODE HERE
    # self_contained_score = coherence * length_factor  # YOUR CODE HERE

    # return {
    #     'coherence': coherence,
    #     'min_similarity': min_sim,
    #     'self_contained_score': self_contained_score
    # }
    # ==============================

    raise NotImplementedError("Complete the TODO above by uncommenting and filling in the code!")

In [ ]:
#@title 🎧 Listen: Todo1 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_todo1_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Verification cell for TODO 1
# Uncomment and run after implementing score_chunk_quality

# # Test with a coherent chunk (all about transformers)
# coherent_chunk = ("The transformer architecture uses self-attention to process sequences. "
#                   "Each token attends to every other token in the sequence. "
#                   "This parallel processing is much faster than recurrent approaches.")
#
# # Test with an incoherent chunk (mixed topics)
# incoherent_chunk = ("The transformer uses self-attention mechanisms. "
#                     "Bananas are a popular fruit in tropical regions. "
#                     "The stock market closed at record highs yesterday.")
#
# score_good = score_chunk_quality(coherent_chunk, embed_model)
# score_bad = score_chunk_quality(incoherent_chunk, embed_model)
#
# print("Coherent chunk scores:")
# for k, v in score_good.items():
#     print(f"  {k}: {v:.4f}")
#
# print("\nIncoherent chunk scores:")
# for k, v in score_bad.items():
#     print(f"  {k}: {v:.4f}")
#
# assert score_good['coherence'] > score_bad['coherence'], \
#     "Coherent chunk should have higher coherence than incoherent chunk!"
# assert score_good['self_contained_score'] > score_bad['self_contained_score'], \
#     "Coherent chunk should have higher self-contained score!"
#
# print("\n All assertions passed! Your chunk quality scorer works correctly.")

In [ ]:
#@title 🎧 Listen: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 🔧 TODO 2: Adaptive Chunking — Content-Aware Chunk Sizes

Instead of using a fixed chunk size everywhere, adapt the chunk size based on the **complexity** of the content. Dense technical text (lots of jargon, unique terms, named entities) should get smaller chunks for precision. Narrative text (flowing prose, repeated vocabulary) should get larger chunks to preserve context.

We measure complexity using a simple proxy: **lexical density** — the ratio of unique words to total words in a sliding window. High lexical density = many unique terms = complex/technical content.

In [ ]:
#@title 🎧 Listen: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def chunk_adaptive(text: str, base_chunk_size: int = 500,
                   min_chunk_size: int = 150, max_chunk_size: int = 800,
                   complexity_window: int = 100) -> List[str]:
    """
    Adaptive chunking: adjust chunk size based on content complexity.

    Dense/technical text gets smaller chunks (more precise retrieval).
    Narrative/flowing text gets larger chunks (more context preserved).

    Args:
        text: The document text to chunk
        base_chunk_size: Starting chunk size in words
        min_chunk_size: Minimum allowed chunk size
        max_chunk_size: Maximum allowed chunk size
        complexity_window: Window size (in words) for computing complexity

    Returns:
        List of text chunks with varying sizes

    Hints:
        1. Split text into sentences
        2. For each position, compute a complexity score using a sliding
           window of nearby sentences:
           - Lexical density = unique_words / total_words in window
           - Higher density = more complex text = smaller chunks
        3. Map complexity score to chunk size:
           - High complexity (density > 0.7) -> min_chunk_size
           - Low complexity (density < 0.4) -> max_chunk_size
           - Linear interpolation in between
        4. Accumulate sentences until the adaptive target size is reached,
           then start a new chunk
    """
    # ============ TODO ============
    # Step 1: Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    # Step 2: Compute lexical density for each sentence position
    # def compute_lexical_density(sentence_group):
    #     """Ratio of unique words to total words."""
    #     words = " ".join(sentence_group).lower().split()
    #     if not words:
    #         return 0.5
    #     return len(set(words)) / len(words)

    # Step 3: For each sentence, determine the target chunk size
    # based on the complexity of the surrounding text
    # densities = []
    # for i in range(len(sentences)):
    #     window_start = max(0, i - 2)
    #     window_end = min(len(sentences), i + 3)
    #     density = compute_lexical_density(sentences[window_start:window_end])
    #     densities.append(density)

    # Step 4: Map density to chunk size (higher density -> smaller chunks)
    # def density_to_chunk_size(density):
    #     # Linear interpolation: density 0.4 -> max_chunk_size, density 0.7 -> min_chunk_size
    #     t = np.clip((density - 0.4) / (0.7 - 0.4), 0, 1)
    #     return int(max_chunk_size - t * (max_chunk_size - min_chunk_size))

    # Step 5: Build chunks using adaptive sizes
    # chunks = []
    # current_chunk = []
    # current_size = 0
    # current_target = base_chunk_size
    #
    # for i, sentence in enumerate(sentences):
    #     s_len = len(sentence.split())
    #     current_target = density_to_chunk_size(densities[i])
    #
    #     if current_size + s_len > current_target and current_chunk:
    #         chunks.append(" ".join(current_chunk))
    #         current_chunk = []
    #         current_size = 0
    #
    #     current_chunk.append(sentence)
    #     current_size += s_len
    #
    # if current_chunk:
    #     chunks.append(" ".join(current_chunk))
    #
    # return chunks
    # ==============================

    raise NotImplementedError("Complete the TODO above by uncommenting and filling in the code!")

In [ ]:
#@title 🎧 Listen: Todo2 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_23_todo2_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Verification cell for TODO 2
# Uncomment and run after implementing chunk_adaptive

# adaptive_chunks = chunk_adaptive(raw_text)
#
# print(f"Adaptive chunking results:")
# print(f"  Total chunks: {len(adaptive_chunks)}")
# chunk_sizes_adp = [len(c.split()) for c in adaptive_chunks]
# print(f"  Size range: {min(chunk_sizes_adp)} to {max(chunk_sizes_adp)} words")
# print(f"  Mean size: {np.mean(chunk_sizes_adp):.0f} words")
# print(f"  Std dev: {np.std(chunk_sizes_adp):.0f} words")
#
# # The std dev should be higher than fixed-size chunking
# # because adaptive chunks vary in size based on content complexity
# fixed_std = np.std([len(c.split()) for c in fixed_chunks])
# print(f"\n  Fixed-size std dev: {fixed_std:.0f}")
# print(f"  Adaptive std dev:  {np.std(chunk_sizes_adp):.0f}")
#
# if np.std(chunk_sizes_adp) > fixed_std * 0.5:
#     print("  Adaptive chunks show meaningful size variation!")
# else:
#     print("  Warning: adaptive chunks are too uniform — check your complexity scoring")
#
# # Visualize size variation
# plt.figure(figsize=(12, 4))
# plt.bar(range(len(chunk_sizes_adp)), chunk_sizes_adp, color='#9C27B0', alpha=0.8)
# plt.xlabel('Chunk Index')
# plt.ylabel('Words per Chunk')
# plt.title('Adaptive Chunking: Variable Chunk Sizes Based on Content Complexity',
#           fontweight='bold')
# plt.tight_layout()
# plt.show()

In [ ]:
#@title 🎧 Listen: Putting It All Together
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_24_putting_it_all_together.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. 🔗 Putting It All Together

Let us run all four strategies on the same document, then compare retrieval quality with the same query.

In [ ]:
# Collect all chunking results
all_strategies = {
    'Fixed-Size (500w, 50 overlap)': fixed_chunks,
    'Sentence-Level (max 500w)': sentence_chunks,
    'Recursive (LangChain)': recursive_chunks,
    'Semantic (threshold=0.35)': semantic_chunks,
}

# Summary statistics
print("=" * 70)
print("CHUNKING STRATEGY COMPARISON")
print("=" * 70)
print(f"{'Strategy':<32} {'Chunks':>7} {'Avg Size':>9} {'Min':>5} {'Max':>5} {'Std':>5}")
print("-" * 70)

for name, chunks in all_strategies.items():
    sizes = [len(c.split()) for c in chunks]
    print(f"{name:<32} {len(chunks):>7} {np.mean(sizes):>8.0f}w {min(sizes):>5} {max(sizes):>5} {np.std(sizes):>5.0f}")

In [ ]:
#@title 🎧 Listen: Viz All Chunk Sizes
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_25_viz_all_chunk_sizes.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 4-panel visualization: chunk size distributions for all strategies
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
strategy_names = list(all_strategies.keys())

for idx, (name, chunks) in enumerate(all_strategies.items()):
    ax = axes[idx // 2][idx % 2]
    sizes = [len(c.split()) for c in chunks]

    # Bar chart of chunk sizes
    ax.bar(range(len(sizes)), sizes, color=colors[idx], alpha=0.8,
           edgecolor='white', width=0.8)
    ax.axhline(y=np.mean(sizes), color='red', linestyle='--', alpha=0.6,
               label=f'Mean = {np.mean(sizes):.0f}w')
    ax.set_xlabel('Chunk Index', fontsize=10)
    ax.set_ylabel('Words', fontsize=10)
    ax.set_title(f'{name}\n({len(chunks)} chunks)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('All Four Chunking Strategies: Chunk Size Comparison',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Retrieval Comparison Single Query
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_26_retrieval_comparison_single_query.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Retrieval comparison: same query, four strategies
query = "How does self-attention work in transformers?"
print(f"Query: \"{query}\"")
print("=" * 70)

query_embedding = embed_model.encode(query)

for name, chunks in all_strategies.items():
    # Embed chunks
    chunk_embeddings = embed_model.encode(chunks, show_progress_bar=False)

    # Compute cosine similarities
    similarities = np.dot(chunk_embeddings, query_embedding) / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_embedding) + 1e-10
    )

    # Get top 3
    top_indices = np.argsort(similarities)[::-1][:3]

    print(f"\n--- {name} ---")
    for rank, idx in enumerate(top_indices, 1):
        chunk_preview = chunks[idx][:120].replace('\n', ' ')
        print(f"  {rank}. [sim={similarities[idx]:.4f}] \"{chunk_preview}...\"")
        print(f"     ({len(chunks[idx].split())} words)")

In [ ]:
#@title 🎧 Listen: Training And Results Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_27_training_and_results_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. 📊 Training and Results

Let us benchmark all four strategies with a proper retrieval evaluation. We define 10 test queries with manually labeled ground truth (a distinctive phrase from the correct answer), then measure Recall@5: what fraction of the time is the correct chunk in the top-5 retrieved chunks?

In [ ]:
# Define test queries and the ground truth text they should retrieve
# Each ground truth is a distinctive phrase that should appear in the correct chunk
test_queries_and_ground_truth = [
    ("How does self-attention work?",
     "query vector, a key vector, and a value vector"),
    ("What is multi-head attention?",
     "jointly attend to information from different representation subspaces"),
    ("How are positions encoded in transformers?",
     "positional encodings"),
    ("What is the feed-forward network in transformers?",
     "two linear transformations with a ReLU activation"),
    ("What task was the transformer originally designed for?",
     "sequence-to-sequence tasks such as machine translation"),
    ("What training method do transformers use?",
     "teacher forcing"),
    ("What is BERT?",
     "Bidirectional Encoder Representations"),
    ("How do Vision Transformers work?",
     "splitting images into patches"),
    ("What are transformer scaling laws?",
     "scaling laws"),
    ("What is FlashAttention?",
     "tiling and recomputation"),
]

def evaluate_retrieval(chunks: List[str], model, queries_gt: list, top_k: int = 5) -> Tuple[float, List[bool]]:
    """
    Evaluate retrieval quality using Recall@K.

    For each query, check if any of the top-K retrieved chunks
    contains the ground truth text.

    Returns:
        Tuple of (recall_score, per_query_hits)
    """
    chunk_embeddings = model.encode(chunks, show_progress_bar=False)
    hits = []

    for query, ground_truth in queries_gt:
        query_emb = model.encode(query)
        similarities = np.dot(chunk_embeddings, query_emb) / (
            np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_emb) + 1e-10
        )
        top_indices = np.argsort(similarities)[::-1][:top_k]

        # Check if any top-K chunk contains the ground truth
        hit = False
        for idx in top_indices:
            if ground_truth.lower() in chunks[idx].lower():
                hit = True
                break
        hits.append(hit)

    return sum(hits) / len(hits), hits


# Evaluate all strategies
print("Evaluating retrieval quality (Recall@5) across 10 test queries...")
print("=" * 60)

results = {}
per_query_results = {}
for name, chunks in all_strategies.items():
    recall, hits = evaluate_retrieval(chunks, embed_model, test_queries_and_ground_truth, top_k=5)
    results[name] = recall
    per_query_results[name] = hits
    print(f"  {name:<35} Recall@5 = {recall:.1%}  ({sum(hits)}/{len(hits)} queries)")

print("=" * 60)
best_strategy = max(results, key=results.get)
print(f"\nBest strategy: {best_strategy} ({results[best_strategy]:.1%})")

In [ ]:
#@title 🎧 Listen: Viz Recall At 5
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_28_viz_recall_at_5.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Bar chart comparison of Recall@5
fig, ax = plt.subplots(figsize=(12, 6))

strategy_labels = list(results.keys())
recall_values = [results[name] for name in strategy_labels]
bar_colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

bars = ax.bar(range(len(strategy_labels)), recall_values,
              color=bar_colors, alpha=0.85, edgecolor='white', width=0.6)

# Add value labels on bars
for bar, val in zip(bars, recall_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.0%}', ha='center', va='bottom', fontweight='bold', fontsize=13)

ax.set_xticks(range(len(strategy_labels)))
ax.set_xticklabels([name.replace(' (', '\n(') for name in strategy_labels],
                    fontsize=10, ha='center')
ax.set_ylabel('Recall@5', fontsize=13)
ax.set_title('Retrieval Quality by Chunking Strategy\n(10 test queries, Recall@5)',
             fontsize=15, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print per-query analysis
print("\nPer-query analysis:")
print("-" * 80)
query_labels = [q[0][:45] for q in test_queries_and_ground_truth]
print(f"{'Query':<47} " + " ".join(f"{'S'+str(i+1):>4}" for i in range(4)))
print("-" * 80)
for qi, ql in enumerate(query_labels):
    row = f"{ql:<47} "
    for si, name in enumerate(all_strategies.keys()):
        hit = per_query_results[name][qi]
        row += f" {'HIT':>4}" if hit else f"{'MISS':>5}"
    print(row)

diff = max(recall_values) - min(recall_values)
print(f"\nGap between best and worst: {diff:.0%}")
print("Chunking strategy choice significantly impacts retrieval quality!")

In [ ]:
#@title 🎧 Listen: Interactive Explorer Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_29_interactive_explorer_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. 🎯 Final Output: Interactive Chunk Explorer

Let us build an interactive chunk explorer that lets you experiment with different strategies and parameters on any text, visualize the chunks, and run test queries.

In [ ]:
def chunk_explorer(text: str, strategy: str = 'fixed',
                   chunk_size: int = 500, overlap: int = 50,
                   threshold: float = 0.35, query: str = None):
    """
    Interactive chunk explorer: chunk text with any strategy,
    visualize the results, and optionally run a test query.

    Args:
        text: Input text to chunk
        strategy: 'fixed', 'sentence', 'recursive', or 'semantic'
        chunk_size: Target chunk size (words for fixed/sentence, chars for recursive)
        overlap: Overlap amount (words for fixed, chars for recursive)
        threshold: Similarity threshold (only for semantic strategy)
        query: Optional query to test retrieval
    """
    # Apply the chosen strategy
    print(f"Strategy: {strategy.upper()}")
    print(f"Parameters: chunk_size={chunk_size}, overlap={overlap}, threshold={threshold}")
    print("=" * 60)

    if strategy == 'fixed':
        chunks = chunk_fixed_size(text, chunk_size=chunk_size, overlap=overlap)
    elif strategy == 'sentence':
        chunks = chunk_by_sentences(text, max_chunk_tokens=chunk_size)
    elif strategy == 'recursive':
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size * 4,  # Convert words to approx chars
            chunk_overlap=overlap * 4,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        chunks = splitter.split_text(text)
    elif strategy == 'semantic':
        chunks, _ = chunk_semantic(text, embed_model, threshold=threshold)
    else:
        raise ValueError(f"Unknown strategy: {strategy}. Use 'fixed', 'sentence', 'recursive', or 'semantic'.")

    # Display statistics
    sizes = [len(c.split()) for c in chunks]
    print(f"Total chunks: {len(chunks)}")
    print(f"Size range: {min(sizes)} to {max(sizes)} words")
    print(f"Mean size: {np.mean(sizes):.0f} words (std: {np.std(sizes):.0f})")

    # Visualization: color-coded chunk boundaries
    fig, axes = plt.subplots(2, 1, figsize=(14, 6),
                              gridspec_kw={'height_ratios': [1, 2]})

    # Top: chunk size bar chart
    ax1 = axes[0]
    chunk_colors = plt.cm.Set3(np.linspace(0, 1, max(len(chunks), 1)))
    ax1.bar(range(len(sizes)), sizes, color=chunk_colors[:len(sizes)],
            edgecolor='white', width=0.8)
    ax1.set_ylabel('Words', fontsize=10)
    ax1.set_title(f'Chunk Sizes ({strategy.capitalize()} Strategy)', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.2, axis='y')

    # Bottom: document map with chunk boundaries
    ax2 = axes[1]
    total_chars = max(sum(len(c) for c in chunks), 1)
    x_pos = 0
    for i, chunk in enumerate(chunks):
        width = len(chunk) / total_chars
        ax2.barh(0, width, left=x_pos, height=0.5,
                 color=chunk_colors[i % len(chunk_colors)], edgecolor='white', linewidth=0.5)
        # Label if wide enough
        if width > 0.05:
            label = f'C{i+1}\n{len(chunk.split())}w'
            ax2.text(x_pos + width/2, 0, label, ha='center', va='center',
                     fontsize=8, fontweight='bold')
        x_pos += width

    ax2.set_xlim(0, 1)
    ax2.set_ylim(-0.5, 0.5)
    ax2.set_yticks([])
    ax2.set_xlabel('Document Position', fontsize=10)
    ax2.set_title('Document Map: Each Color = One Chunk', fontsize=11)

    plt.tight_layout()
    plt.show()

    # Run test query if provided
    if query:
        print(f"\nTest Query: \"{query}\"")
        print("-" * 50)
        chunk_embs = embed_model.encode(chunks, show_progress_bar=False)
        query_emb = embed_model.encode(query)
        sims = np.dot(chunk_embs, query_emb) / (
            np.linalg.norm(chunk_embs, axis=1) * np.linalg.norm(query_emb) + 1e-10
        )
        top_k = min(3, len(chunks))
        top_idx = np.argsort(sims)[::-1][:top_k]

        for rank, idx in enumerate(top_idx, 1):
            preview = chunks[idx][:150].replace('\n', ' ')
            print(f"  {rank}. [sim={sims[idx]:.4f}] Chunk {idx+1}: \"{preview}...\"")

    return chunks


# Demo: explore different strategies with the same query
print("DEMO 1: Fixed-size chunking with smaller chunks")
print("=" * 60)
_ = chunk_explorer(raw_text, strategy='fixed', chunk_size=200, overlap=20,
                   query="How does self-attention work?")

In [ ]:
#@title 🎧 Listen: Interactive Explorer Demos
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_30_interactive_explorer_demos.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Demo 2: Sentence-level chunking
print("DEMO 2: Sentence-level chunking")
print("=" * 60)
_ = chunk_explorer(raw_text, strategy='sentence', chunk_size=200,
                   query="How does self-attention work?")

In [ ]:
# Demo 3: Semantic chunking
print("DEMO 3: Semantic chunking")
print("=" * 60)
_ = chunk_explorer(raw_text, strategy='semantic', threshold=0.35,
                   query="How does self-attention work?")

In [ ]:
# Try it yourself! Modify these parameters and re-run:

# Option 1: Use the Wikipedia text we already loaded
your_text = raw_text

# Option 2: Fetch a different Wikipedia article — just change the title!
# wiki_url = "https://en.wikipedia.org/w/api.php"
# wiki_params = {"action": "query", "titles": "Reinforcement learning",
#                "prop": "extracts", "explaintext": True, "format": "json"}
# wiki_response = requests.get(wiki_url, params=wiki_params)
# wiki_data = wiki_response.json()
# wiki_pages = wiki_data['query']['pages']
# your_text = list(wiki_pages.values())[0].get('extract', '')

# Experiment with these parameters:
_ = chunk_explorer(
    your_text,
    strategy='fixed',     # Try: 'fixed', 'sentence', 'recursive', 'semantic'
    chunk_size=300,       # Adjust chunk size
    overlap=30,           # Adjust overlap (fixed/recursive only)
    threshold=0.4,        # Adjust threshold (semantic only)
    query="What is the most important innovation in the transformer?"
)

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_31_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. 🤔 Reflection and Next Steps

### Reflection Questions

Take a moment to think through these questions. They test whether you have internalized the key ideas, not just memorized the code.

**Question 1: The Optimal Chunk Size**

You are building a RAG system for a legal document corpus (contracts, regulations, case law). Documents are dense, with precise language where every word matters. Would you use smaller or larger chunks than the typical 500-token recommendation? Why? What is the specific trade-off you are making?

*Think about: information density, precision vs context, the probability of splitting a key clause at a boundary.*

**Question 2: The Overlap Tax**

Increasing overlap from 0% to 20% of chunk size typically improves boundary preservation. But what is the cost? If your corpus has 1 million chunks at 500 tokens with 0% overlap, how many chunks would you have at 20% overlap ($O = 100$)? Use the formula: $K = \lceil (N - O) / (C - O) \rceil$. How does this affect storage, embedding compute cost, and search latency?

*Think about: the formula, and the downstream effects on the vector database.*

**Question 3: Is Semantic Chunking Worth the Compute Cost?**

Semantic chunking requires embedding every sentence in the document (not just each chunk), which is roughly $N_{\text{sentences}} / N_{\text{chunks}}$ times more expensive at indexing time. Under what conditions is this extra cost justified? When would sentence-level chunking be "good enough"?

*Think about: document homogeneity, topic diversity within documents, the value of retrieval precision in your specific application, and whether your documents have natural structure (headers, sections) that recursive chunking can exploit.*

### Optional Challenges

**Challenge 1: Hierarchical Chunk Retrieval**

Implement a two-level chunking system: large "parent" chunks (1000 tokens) and small "child" chunks (200 tokens). At retrieval time, search over child chunks for precision, but return the parent chunk for context. This gives you the best of both worlds — precise matching with full context. This is the approach used by LlamaIndex's `ParentDocumentRetriever`.

**Challenge 2: Document-Aware Chunking**

Implement a chunker that uses document structure (headings, bullet points, code blocks) as primary split points. Parse the document's Markdown or HTML structure and use headers as natural chunk boundaries, falling back to sentence-level splitting within each section. Test whether structure-aware chunking outperforms blind recursive chunking on a structured document like a Wikipedia article.

---

**What is next?** In the next notebook, we will tackle **Embeddings and Vector Search** — how to turn chunks into vectors and build efficient retrieval indexes. The quality of your chunks determines the ceiling; the quality of your embeddings and search determines how close you get to it.